# DnCNN baseline for the denoising table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Theborna/symmetric_parseval_conv/blob/main/colab_dncnn_baseline.ipynb)

This trains a plain, **unconstrained** DnCNN at the same noise levels
($\sigma = 5, 15, 25$) and a comparable depth/width to the four Parseval
models, so you can add it to the table from `colab_experiments.ipynb` as one
more row. It does not modify that notebook or its saved results.

**Why DnCNN specifically:** it is the standard denoising-CNN baseline with no
architectural constraint on its Jacobian — unlike every model in
`parseval_cnn.py`, it is not 1-Lipschitz by construction. It answers: how much
denoising accuracy does the Lipschitz constraint cost?

**Fair-comparison choices:**
* **Depth/width** — `depth=16, nb_channels=64`, matching `BaselineParsevalCNN`
  / `SymmetricParsevalCNN`'s convolution-layer count and channel width exactly
  (the nested `mirror` / `symmetric_mirror` models use `depth=8`, but each
  level adds two convolutions, so 16 conv ops either way). Section 4 below
  prints the actual parameter counts side by side so you can see how close
  this is, rather than taking "similarly sized" on faith.
* **Training budget** — same `sigma` values, same epochs/batch size as the main
  sweep (`experiment_configs/baseline.json`). Only the learning rate differs
  (`1e-3` instead of `1e-5`): the Parseval models' rate is tuned for
  Björck-orthogonalized weights and is far too small to train a plain CNN in
  the same number of epochs, so using it here would not be a fair comparison
  either.

Runtime: *Runtime → Change runtime type → GPU* before you start.


## 1. Check the runtime


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4 is fine).')

## 2. Get the code


In [ ]:
import os

REPO_DIR = 'symmetric_parseval_conv'
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    !git clone https://github.com/Theborna/symmetric_parseval_conv.git
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)
!git pull --ff-only
print('Working dir:', os.getcwd())

## 3. Install dependencies


In [ ]:
!pip install -q tqdm tensorboard h5py einops scikit-image matplotlib piqa pytorch-ssim

# utils/utilities.py imports pytorch_ssim at module load; make sure it resolves.
try:
    import pytorch_ssim, piqa  # noqa: F401
    print('SSIM deps OK')
except Exception as e:
    print('installing pytorch_ssim from source:', e)
    !pip install -q git+https://github.com/Po-Hsun-Su/pytorch-ssim.git

## 4. Sanity check: is DnCNN actually "similarly sized"?

Instantiates `DnCNNBaseline` alongside the Parseval models at the depth/width
this notebook trains with, and prints their parameter counts side by side.


In [ ]:
from parseval_cnn import MODELS

DEPTH, WIDTH, KSIZE = 16, 64, 3
act_dummy = {'spline_size': 51, 'spline_range': 0.1, 'lmbda': 1e-6, 'entro': 0.1}

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'{"model":20s} {"depth (conv layers)":22s} {"params":>12s}')
for key, d in [('baseline', DEPTH), ('symmetric', DEPTH), ('mirror', DEPTH // 2),
               ('symmetric_mirror', DEPTH // 2), ('dncnn', DEPTH)]:
    net_params = {'depth': d, 'nb_channels': WIDTH, 'kernel_size': KSIZE, 'bias': False}
    m = MODELS[key](net_params, act_dummy)
    n_conv = DEPTH if key in ('baseline', 'symmetric', 'dncnn') else DEPTH  # 2 per nested level
    print(f'{key:20s} {n_conv:<22d} {count_params(m):>12,d}')
    del m

## 5. Get your BSD500 data onto Colab

Same four options as `colab_experiments.ipynb`. Option B below is pre-filled
with the GitHub Release URLs already used there — if you set those up, this
just works; otherwise pick whichever option suits you (all land the files in
`/content/data/`).


### Option A — upload straight from your computer

Zero setup. Reliable for files up to a few hundred MB; slow/flaky for multi-GB
files (use Option B/C for those). A dialog will ask you to pick both files.


In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
from google.colab import files
print('Select train.h5 and test.h5 ...')
uploaded = files.upload()
for fn in uploaded:
    os.replace(fn, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

### Option B — download from a direct URL

Pre-filled with the same GitHub Release asset used in `colab_experiments.ipynb`.
Works with any direct link: Dropbox (append `?dl=1`), OneDrive, a personal /
university server, or a GitHub Release asset on your own repo (up to 2 GB per
file).


In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
TRAIN_URL = 'https://github.com/Theborna/symmetric_parseval_conv/releases/download/data/train.h5'
VAL_URL   = 'https://github.com/Theborna/symmetric_parseval_conv/releases/download/data/test.h5'
assert TRAIN_URL and VAL_URL, 'set TRAIN_URL and VAL_URL first'
!wget -q --show-progress -O /content/data/train.h5 "{TRAIN_URL}"
!wget -q --show-progress -O /content/data/test.h5  "{VAL_URL}"
print('saved:', os.listdir('/content/data'))

### Option C — Hugging Face Hub (durable, good for reruns)

Upload the two files once to a (private) dataset repo, then pull them here.


In [ ]:
!pip install -q huggingface_hub
import os, shutil
from huggingface_hub import hf_hub_download  # , login

# login('hf_xxx')          # uncomment for a PRIVATE dataset repo
HF_REPO = 'your-username/bsd500'   # <-- your dataset repo id

os.makedirs('/content/data', exist_ok=True)
for fn in ['train.h5', 'test.h5']:
    p = hf_hub_download(repo_id=HF_REPO, filename=fn, repo_type='dataset')
    shutil.copy(p, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

### Option D — Google Drive

Only if you do have Drive access on this account.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# then set TRAIN_H5/VAL_H5 below to your Drive paths

### Set the paths and patch *only* `experiment_configs/dncnn.json` (required)

Run this after whichever option above you used. Unlike `colab_experiments.ipynb`
(which patches every config), this only touches `dncnn.json` — the other
configs, and anything you already have under `exps/`, are left alone.


In [ ]:
import json

# Options A/B/C save here; for Drive (D) point these at your Drive paths.
TRAIN_H5 = '/content/data/train.h5'
VAL_H5   = '/content/data/test.h5'

BATCH_SIZE = 480    # match whatever you used for the main sweep, for a fair comparison
NUM_WORKERS = 2

assert os.path.exists(TRAIN_H5), f'train file not found: {TRAIN_H5}'
assert os.path.exists(VAL_H5),   f'val file not found: {VAL_H5}'

CFG_PATH = 'experiment_configs/dncnn.json'
with open(CFG_PATH) as f:
    cfg = json.load(f)
cfg['training_options']['train_data_file'] = TRAIN_H5
cfg['training_options']['val_data_file'] = VAL_H5
cfg['training_options']['batch_size'] = BATCH_SIZE
cfg['training_options']['num_workers'] = NUM_WORKERS
with open(CFG_PATH, 'w') as f:
    json.dump(cfg, f, indent=4)
print('patched', CFG_PATH)

## 6. Train

Reuses `experiments.py` exactly as `colab_experiments.ipynb` does (same
training/eval code, same checkpointing), restricted to the `dncnn` config.
Writes to its own output directory (`exps/dncnn_baseline`), so it can never
overwrite the main sweep's results even if run in the same session.


In [ ]:
EPOCHS = 10        # match the main sweep's epoch count for a fair comparison
OUTPUT = 'exps/dncnn_baseline'

!python experiments.py -d {device} --configs dncnn -o {OUTPUT} --epochs {EPOCHS}

## 7. The row to add to your table


In [ ]:
import re
import experiments as E

results = json.load(open(os.path.join(OUTPUT, 'results.json')))
sigmas = E._all_sigmas(results, ['dncnn'])

print('Markdown row (paste under the header separator in your existing table):\n')
md_row = E.make_markdown(results, ['dncnn'], sigmas, 'best').strip().split('\n')[-1]
print(md_row)

tex = E.make_latex(results, ['dncnn'], sigmas, 'best')
lines = tex.split('\n')
row = lines[lines.index('\\midrule') + 1]
# With only one model in this table, make_latex trivially bolds every value
# (it is comparing dncnn against itself) -- strip that before pasting; bold
# whichever entry actually wins once this sits alongside the other four rows.
row = re.sub(r'\\textbf\{([^}]*)\}', r'\1', row)

print('\nLaTeX row (paste as one more row inside your existing tabular, before')
print('\\bottomrule; re-bold whichever value is actually best per column):\n')
print(row)

## 8. (Optional) Merge straight into an existing table

If `exps/paper/results.json` already exists in *this* runtime (e.g. you ran
`colab_experiments.ipynb` in the same session), this cell merges the `dncnn`
row into it and regenerates the full combined table — no copy-pasting needed.
Skip this cell if that file isn't here; section 7 above already gives you
everything to add the row by hand.


In [ ]:
MAIN_OUTPUT = 'exps/paper'
main_results_path = os.path.join(MAIN_OUTPUT, 'results.json')

if os.path.exists(main_results_path):
    with open(main_results_path) as f:
        main_results = json.load(f)
    main_results['dncnn'] = results['dncnn']
    with open(main_results_path, 'w') as f:
        json.dump(main_results, f, indent=2, sort_keys=True)

    order = [k for k in MODELS if k in main_results]
    E.write_tables(main_results, order, 'best', MAIN_OUTPUT)
    print(f'\nMerged into {main_results_path} and regenerated the combined table above.')
else:
    print(f'{main_results_path} not found in this runtime -- nothing to merge.')
    print('Use the standalone row from section 7 instead.')